# Install Dependencies

In [1]:
!pip install --quiet pypdf sentence-transformers faiss-cpu pandas matplotlib numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 54.2 MB/s eta 0:00:00


# Imports & Global Config

In [ ]:

import os
from dataclasses import dataclass
from typing import Any, List, Tuple
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

from src.config import (
    DEFAULT_CORPUS_PATH,
    SMALL_EMBEDDING_MODEL_NAME,
    LARGE_EMBEDDING_MODEL_NAME,
    VECTOR_DIM_SMALL,
    VECTOR_DIM_LARGE,
    EVAL_EMBEDDING_MODEL_NAME,
    EXPERIMENT_TOP_K,
)
from src.data_ingest import load_pdf_text
from src.chunking import FixedSizeChunker, StructureAwareChunker
from src.embeddings import EmbeddingModel
from src.vector_store import FaissVectorStore, DocumentChunk
from src.memory import STMMemory, LTMMemory, EntityMemory
from src.agent import RAGAgent, SimpleHeuristicGenerator
from src.data_model import load_default_questions, QuestionItem
from src.evaluation import (
    build_gold_chunk_mapping,
    evaluate_agent_on_questions,
    aggregate_results,
)

np.random.seed(42)

# Implementation

In [3]:


def main():
    os.makedirs("artifacts", exist_ok=True)

    # -----------------------------
    # 1. Load thesis text
    # -----------------------------
    corpus_path = DEFAULT_CORPUS_PATH
    print("Loading thesis PDF from:", corpus_path)
    text = load_pdf_text(corpus_path)
    print("Loaded chars:", len(text))
    print(text[:1000], "\n---\n")

    # -----------------------------
    # 2. Base chunking (structure-aware)
    # -----------------------------
    struct_chunker = StructureAwareChunker(min_chunk_chars=400, max_chunk_chars=1600)
    struct_chunks = struct_chunker.chunk(text)
    base_docs: List[DocumentChunk] = [
        DocumentChunk(id=c["id"], text=c["text"], metadata=c["metadata"])
        for c in struct_chunks
    ]
    print("Structure-aware chunks:", len(base_docs))

    # -----------------------------
    # 3. Embeddings & vector store
    # -----------------------------
    small_embed = EmbeddingModel(SMALL_EMBEDDING_MODEL_NAME)
    eval_model = SentenceTransformer(EVAL_EMBEDDING_MODEL_NAME)

    base_vs = FaissVectorStore(dimension=small_embed.dimension)
    base_vs.build_index(
        docs=base_docs,
        embed_fn=small_embed.encode_texts,
    )

    # -----------------------------
    # 4. Memory & agent (base config)
    # -----------------------------
    stm = STMMemory(max_tokens=800)
    ltm = LTMMemory(json_path="artifacts/ltm_memory_base.json", embedding_model=small_embed)
    entities = EntityMemory()

    # Seed some known entities
    entities.upsert_entity("Zainab Saad", "student", {"id": "202472448"})
    entities.upsert_entity("Ibrahim Issa", "advisor", {"role": "Thesis Advisor"})
    entities.upsert_entity("Khalil Hariss", "advisor", {"role": "Co-Advisor"})
    entities.upsert_entity("Razane Tajeddine", "committee", {"role": "Committee Member"})
    entities.upsert_entity("Enron Email dataset", "dataset", {"domain": "corporate email"})
    entities.upsert_entity("HealthcareMagic-101", "dataset", {"domain": "healthcare Q&A"})
    entities.upsert_entity("DP-MLM", "dp_mechanism", {"description": "DP masked LM"})
    entities.upsert_entity("DP-BART", "dp_mechanism", {"description": "DP text rewriting model"})

    generator = SimpleHeuristicGenerator(eval_model=eval_model)
    base_agent = RAGAgent(
        vector_store=base_vs,
        embed_model=small_embed,
        stm=stm,
        ltm=ltm,
        entities=entities,
        generator=generator,
        top_k=EXPERIMENT_TOP_K,
        use_ltm=True,
    )

    # Quick sanity check
    q = "What is the main research question of the thesis?"
    ans, debug = base_agent.answer(q)
    print("Q:", q)
    print("A:", ans[:500], "\n")

    # -----------------------------
    # 5. Questions & gold mapping
    # -----------------------------
    questions: List[QuestionItem] = load_default_questions()
    gold_mapping = build_gold_chunk_mapping(
        questions=questions,
        docs=base_docs,
        eval_embed_model=eval_model,
    )

    # -----------------------------
    # 6. Experiment configs
    # -----------------------------

    @dataclass
    class ExperimentConfig:
        name: str
        use_structure_chunking: bool
        fixed_chunk_size: int  # in chars
        fixed_overlap: int
        embedding_model_name: str
        memory_policy: str  # "stm-only" or "stm-ltm"

    experiment_configs: List[ExperimentConfig] = []

    # A: Chunk size (small vs large)
    experiment_configs += [
        ExperimentConfig(
            name="A_small_chunk",
            use_structure_chunking=False,
            fixed_chunk_size=500,
            fixed_overlap=100,
            embedding_model_name=SMALL_EMBEDDING_MODEL_NAME,
            memory_policy="stm-ltm",
        ),
        ExperimentConfig(
            name="A_large_chunk",
            use_structure_chunking=False,
            fixed_chunk_size=1500,
            fixed_overlap=200,
            embedding_model_name=SMALL_EMBEDDING_MODEL_NAME,
            memory_policy="stm-ltm",
        ),
    ]

    # B: Chunking strategy (fixed vs structure)
    experiment_configs += [
        ExperimentConfig(
            name="B_fixed_chunking",
            use_structure_chunking=False,
            fixed_chunk_size=500,
            fixed_overlap=100,
            embedding_model_name=SMALL_EMBEDDING_MODEL_NAME,
            memory_policy="stm-ltm",
        ),
        ExperimentConfig(
            name="B_structure_chunking",
            use_structure_chunking=True,
            fixed_chunk_size=500,
            fixed_overlap=100,
            embedding_model_name=SMALL_EMBEDDING_MODEL_NAME,
            memory_policy="stm-ltm",
        ),
    ]

    # C: Embeddings (small vs large)
    experiment_configs += [
        ExperimentConfig(
            name="C_small_embedding",
            use_structure_chunking=False,
            fixed_chunk_size=500,
            fixed_overlap=100,
            embedding_model_name=SMALL_EMBEDDING_MODEL_NAME,
            memory_policy="stm-ltm",
        ),
        ExperimentConfig(
            name="C_large_embedding",
            use_structure_chunking=False,
            fixed_chunk_size=500,
            fixed_overlap=100,
            embedding_model_name=LARGE_EMBEDDING_MODEL_NAME,
            memory_policy="stm-ltm",
        ),
    ]

    # D: Memory policy (STM-only vs STM+LTM)
    experiment_configs += [
        ExperimentConfig(
            name="D_stm_only",
            use_structure_chunking=False,
            fixed_chunk_size=500,
            fixed_overlap=100,
            embedding_model_name=SMALL_EMBEDDING_MODEL_NAME,
            memory_policy="stm-only",
        ),
        ExperimentConfig(
            name="D_stm_ltm",
            use_structure_chunking=False,
            fixed_chunk_size=500,
            fixed_overlap=100,
            embedding_model_name=SMALL_EMBEDDING_MODEL_NAME,
            memory_policy="stm-ltm",
        ),
    ]

    # -----------------------------
    # 7. Helper to build agent per config
    # -----------------------------

    def make_agent_for_config(
        cfg: ExperimentConfig,
        raw_text: str,
    ) -> Tuple[RAGAgent, List[DocumentChunk], Dict[int, str]]:
        if cfg.use_structure_chunking:
            chunker = StructureAwareChunker(min_chunk_chars=400, max_chunk_chars=1600)
        else:
            chunker = FixedSizeChunker(
                chunk_size_chars=cfg.fixed_chunk_size,
                overlap_chars=cfg.fixed_overlap,
            )

        chunks = chunker.chunk(raw_text)
        docs = [
            DocumentChunk(id=c["id"], text=c["text"], metadata=c["metadata"])
            for c in chunks
        ]

        embed_model = EmbeddingModel(cfg.embedding_model_name)
        vs = FaissVectorStore(dimension=embed_model.dimension)
        vs.build_index(docs=docs, embed_fn=embed_model.encode_texts)

        stm_local = STMMemory(max_tokens=800)
        ltm_json = f"artifacts/ltm_{cfg.name}.json"
        ltm_local = LTMMemory(json_path=ltm_json, embedding_model=embed_model)
        entities_local = EntityMemory()
        generator_local = SimpleHeuristicGenerator(eval_model=eval_model)
        use_ltm = cfg.memory_policy == "stm-ltm"

        agent_local = RAGAgent(
            vector_store=vs,
            embed_model=embed_model,
            stm=stm_local,
            ltm=ltm_local,
            entities=entities_local,
            generator=generator_local,
            top_k=EXPERIMENT_TOP_K,
            use_ltm=use_ltm,
        )

        gold_map_local = build_gold_chunk_mapping(
            questions=questions,
            docs=docs,
            eval_embed_model=eval_model,
        )

        return agent_local, docs, gold_map_local

    # -----------------------------
    # 8. Run all experiments
    # -----------------------------
    all_results: List[pd.DataFrame] = []

    for cfg in experiment_configs:
        print(f"\n=== Running experiment: {cfg.name} ===")
        agent_cfg, docs_cfg, gold_map_cfg = make_agent_for_config(cfg, text)

        metrics_df = evaluate_agent_on_questions(
            agent=agent_cfg,
            questions=questions,
            gold_chunk_mapping=gold_map_cfg,
            eval_embed_model=eval_model,
            top_k=EXPERIMENT_TOP_K,
            experiment_name=cfg.name,
        )
        all_results.append(metrics_df)

    all_results_df = pd.concat(all_results, ignore_index=True)
    all_results_df.to_csv("artifacts/all_experiment_results.csv", index=False)
    print("\nSample per-question results:")
    print(all_results_df.head())

    # -----------------------------
    # 9. Aggregate & save summary
    # -----------------------------
    summary_df = aggregate_results(all_results_df)
    summary_df.to_csv("artifacts/experiment_summary.csv", index=False)
    print("\n=== Experiment Summary ===")
    print(summary_df)

    # -----------------------------
    # 10. Example plot: chunk-size ablation
    # -----------------------------
    try:
        mask = summary_df["experiment_name"].isin(["A_small_chunk", "A_large_chunk"])
        chunk_summary = summary_df[mask]

        plt.figure(figsize=(6, 4))
        plt.bar(chunk_summary["experiment_name"], chunk_summary["hit_rate_at_k"])
        plt.title("Chunk Size Ablation – Hit@k")
        plt.ylabel("Hit@k")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.savefig("artifacts/chunk_size_ablation_hit_at_k.png")
        plt.close()
    except Exception as e:
        print("Plotting failed:", e)


if __name__ == "__main__":
    main()



Loading thesis PDF from: data/proposal_thesis.pdf
Loaded chars: 59533
1 
 
 
 
Graduate Studies 
Masters Proposal Form  
 
Department: ECE  
Major: EECE 
Student’s Name: Zainab Saad   ID:  202472448 
 
Thesis / Project Title: 
 
Deep Comparative Evaluation of Differential Privacy Integration  
 Across Multi-Phase Retrieval Augmented Generation Pipelines 
 
Thesis / Project Advisor:   Prof. Ibrahim Issa 
 
Thesis Co-Advisor: 
 Prof. Khalil Hariss 
Name   Signature  Date 
Prof. Ibrahim Issa      
Prof. Razane Tajeddine      
Prof.       
      
 
Tentative Dates For 
 Comprehensive Exam:     
 Thesis Defense:     
 Graduation:     
 
Date Submitted:   
Revised Date:  
 
 
Approval 
Chairperson of Department / Program: Ali Chehab Date:  
Graduate Studies Committee (Chairperson):    
 
Oct, 23, 2025
October 23, 2025

2 
 
      
  
Department of Electrical and Computer Engineering 
 
Thesis Title 
 
 
Thesis Proposal  
Fall 2022-2023 
 
 
 
 
 
 
 
 
Submitted by: Student name, student ID 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q: What is the main research question of the thesis?
A: Based on the thesis proposal, here is a concise answer:

3 Research Objectives.

Key entities mentioned so far:
Zainab Saad (student): id=202472448
Ibrahim Issa (advisor): role=Thesis Advisor
Khalil Hariss (advisor): role=Co-Advisor
Razane Tajeddine (committee): role=Committee Member
Enron Email dataset (dataset): domain=corporate email
HealthcareMagic-101 (dataset): domain=healthcare Q&A
DP-MLM (dp_mechanism): description=DP masked LM
DP-BART (dp_mechanism): description=DP text rewriting model 


=== Running experiment: A_small_chunk ===

=== Running experiment: A_large_chunk ===

=== Running experiment: B_fixed_chunking ===

=== Running experiment: B_structure_chunking ===

=== Running experiment: C_small_embedding ===

=== Running experiment: C_large_embedding ===

=== Running experiment: D_stm_only ===

=== Running experiment: D_stm_ltm ===

Sample per-question results:
  experiment_name  question_id  hit_at_k  reciprocal_rank